<a href="https://colab.research.google.com/github/2212517-PhanLeMinhPhu/tool-web-quan-trac/blob/main/webtool_quantracthucdia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import json
import csv
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any

class IrrigationDataFilter:
    """Filter irrigation monitoring data by date and time range."""

    def __init__(self, json_file_path: str):
        """
        Initialize the filter with a JSON data file.

        Args:
            json_file_path: Path to the JSON file containing irrigation data
        """
        self.json_file_path = json_file_path
        self.data = []
        self.filtered_data = []
        self.load_data()

    def load_data(self) -> bool:
        """
        Load data from JSON file.

        Returns:
            bool: True if successful, False otherwise
        """
        try:
            with open(self.json_file_path, 'r', encoding='utf-8') as f:
                self.data = json.load(f)
            print(f"✅ Loaded {len(self.data)} records from {self.json_file_path}")
            return True
        except FileNotFoundError:
            print(f"❌ File not found: {self.json_file_path}")
            return False
        except json.JSONDecodeError:
            print(f"❌ Invalid JSON format in {self.json_file_path}")
            return False
        except Exception as e:
            print(f"❌ Error loading file: {str(e)}")
            return False

    def parse_time(self, time_str: str) -> datetime:
        """
        Parse time string in format 'YYYY-MM-DD HH-MM-SS'.

        Args:
            time_str: Time string from data

        Returns:
            datetime: Parsed datetime object
        """
        try:
            # Replace hyphens with colons for time part
            parts = time_str.split()
            if len(parts) == 2:
                date_part = parts[0]
                time_part = parts[1].replace('-', ':')
                time_str = f"{date_part} {time_part}"
            return datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            return None

    def filter_by_date_and_time(self, date_str: str,
                               start_hour: int = 5,
                               end_hour: int = 23) -> List[Dict[str, Any]]:
        """
        Filter data by date and time range (5 AM to 11:59 PM by default).

        Args:
            date_str: Date in format 'YYYY-MM-DD'
            start_hour: Start hour (default: 5 for 5:00 AM)
            end_hour: End hour (default: 23 for 11:59 PM)

        Returns:
            List of filtered records
        """
        self.filtered_data = []

        try:
            target_date = datetime.strptime(date_str, '%Y-%m-%d')
        except ValueError:
            print(f"❌ Invalid date format. Use YYYY-MM-DD")
            return []

        for record in self.data:
            if 'Thời gian' not in record:
                continue

            parsed_time = self.parse_time(record['Thời gian'])

            if parsed_time is None:
                continue

            # Check if date matches
            if parsed_time.date() != target_date.date():
                continue

            # Check if time is within range
            hour = parsed_time.hour
            if start_hour <= hour <= end_hour:
                self.filtered_data.append(record)

        return self.filtered_data

    def get_statistics(self) -> Dict[str, Any]:
        """
        Calculate statistics from filtered data.

        Returns:
            Dictionary with statistics
        """
        if not self.filtered_data:
            return {
                'total_records': 0,
                'stations': set(),
                'avg_temperature': 0,
                'avg_humidity': 0,
                'avg_ec': 0,
                'avg_ph': 0,
                'avg_n': 0,
                'avg_p': 0,
                'avg_k': 0,
            }

        temps = []
        humids = []
        ecs = []
        phs = []
        ns = []
        ps = []
        ks = []
        stations = set()

        for record in self.filtered_data:
            stations.add(record.get('STT', 'Unknown'))

            # Temperature from station 5 (tempKK)
            if 'tempKK' in record:
                try:
                    temps.append(float(record['tempKK']))
                except ValueError:
                    pass

            # Humidity
            for humidity_key in ['humiKK', 'Độ ẩm']:
                if humidity_key in record:
                    try:
                        humids.append(float(record[humidity_key]))
                    except ValueError:
                        pass

            # EC (Electrical Conductivity)
            if 'EC' in record:
                try:
                    ecs.append(float(record['EC']))
                except ValueError:
                    pass

            # pH
            if 'PH' in record:
                try:
                    phs.append(float(record['PH']))
                except ValueError:
                    pass

            # NPK values
            for key, list_ref in [('N', ns), ('P', ps), ('K', ks)]:
                if key in record:
                    try:
                        list_ref.append(float(record[key]))
                    except ValueError:
                        pass

        return {
            'total_records': len(self.filtered_data),
            'stations': sorted(list(stations)),
            'num_stations': len(stations),
            'avg_temperature': round(sum(temps) / len(temps), 2) if temps else 0,
            'avg_humidity': round(sum(humids) / len(humids), 2) if humids else 0,
            'avg_ec': round(sum(ecs) / len(ecs), 2) if ecs else 0,
            'avg_ph': round(sum(phs) / len(phs), 2) if phs else 0,
            'avg_n': round(sum(ns) / len(ns), 2) if ns else 0,
            'avg_p': round(sum(ps) / len(ps), 2) if ps else 0,
            'avg_k': round(sum(ks) / len(ks), 2) if ks else 0,
        }

    def display_results(self, limit: int = None):
        """
        Display filtered results in a formatted table.

        Args:
            limit: Maximum number of records to display (None for all)
        """
        if not self.filtered_data:
            print("\n❌ No records found for the specified date and time range.")
            return

        print("\n" + "="*100)
        print(f"📊 FILTERED IRRIGATION DATA - {len(self.filtered_data)} records found")
        print("="*100)

        # Display statistics
        stats = self.get_statistics()
        print(f"\n📈 Statistics:")
        print(f"   • Total Records: {stats['total_records']}")
        print(f"   • Stations: {', '.join(stats['stations'])}")
        print(f"   • Avg Temperature: {stats['avg_temperature']}°C")
        print(f"   • Avg Humidity: {stats['avg_humidity']}%")
        print(f"   • Avg EC: {stats['avg_ec']}")
        print(f"   • Avg pH: {stats['avg_ph']}")
        print(f"   • Avg NPK: N={stats['avg_n']}, P={stats['avg_p']}, K={stats['avg_k']}")

        # Display table
        print("\n" + "-"*100)
        print(f"{'Time':<20} {'ST':<3} {'Temp(°C)':<12} {'Hum(%)':<12} {'EC':<10} {'pH':<8} {'N':<8} {'P':<8} {'K':<8}")
        print("-"*100)

        display_count = limit if limit else len(self.filtered_data)
        for i, record in enumerate(self.filtered_data[:display_count]):
            time_str = record.get('Thời gian', 'N/A')[:19]
            station = record.get('STT', 'N/A')

            temp = record.get('tempKK', record.get('Nhiệt Độ', 'N/A'))
            hum = record.get('humiKK', record.get('Độ ẩm', 'N/A'))
            ec = record.get('EC', 'N/A')
            ph = record.get('PH', 'N/A')
            n = record.get('N', 'N/A')
            p = record.get('P', 'N/A')
            k = record.get('K', 'N/A')

            print(f"{time_str:<20} {station:<3} {str(temp):<12} {str(hum):<12} {str(ec):<10} {str(ph):<8} {str(n):<8} {str(p):<8} {str(k):<8}")

        if limit and len(self.filtered_data) > limit:
            print(f"\n... and {len(self.filtered_data) - limit} more records")
        print("-"*100)

    def export_to_csv(self, output_file: str) -> bool:
        """
        Export filtered data to CSV file.

        Args:
            output_file: Path to output CSV file

        Returns:
            bool: True if successful, False otherwise
        """
        if not self.filtered_data:
            print("❌ No data to export. Filter data first.")
            return False

        try:
            # Get all unique keys from all records
            all_keys = set()
            for record in self.filtered_data:
                all_keys.update(record.keys())

            all_keys = sorted(list(all_keys))

            with open(output_file, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=all_keys)
                writer.writeheader()
                writer.writerows(self.filtered_data)

            print(f"✅ Successfully exported {len(self.filtered_data)} records to {output_file}")
            return True
        except Exception as e:
            print(f"❌ Error exporting to CSV: {str(e)}")
            return False

    def export_to_json(self, output_file: str) -> bool:
        """
        Export filtered data to JSON file.

        Args:
            output_file: Path to output JSON file

        Returns:
            bool: True if successful, False otherwise
        """
        if not self.filtered_data:
            print("❌ No data to export. Filter data first.")
            return False

        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(self.filtered_data, f, ensure_ascii=False, indent=2)

            print(f"✅ Successfully exported {len(self.filtered_data)} records to {output_file}")
            return True
        except Exception as e:
            print(f"❌ Error exporting to JSON: {str(e)}")
            return False


def main():
    """Main function to demonstrate usage."""

    print("🌾 Irrigation Monitoring Data Filter")
    print("="*50)

    # Initialize filter with your JSON file
    json_file = "Quan trắc thực địa (1).json"

    filter_obj = IrrigationDataFilter(json_file)

    if not filter_obj.data:
        print("Could not load data. Exiting.")
        return

    # Interactive menu
    while True:
        print("\n📋 Menu:")
        print("  1. Filter by date and time")
        print("  2. View statistics")
        print("  3. Export to CSV")
        print("  4. Export to JSON")
        print("  5. Exit")

        choice = input("\nSelect option (1-5): ").strip()

        if choice == '1':
            date_str = input("Enter date (YYYY-MM-DD): ").strip()
            start_hour = input("Enter start hour (default 5): ").strip()
            end_hour = input("Enter end hour (default 23): ").strip()

            start_hour = int(start_hour) if start_hour.isdigit() else 5
            end_hour = int(end_hour) if end_hour.isdigit() else 23

            filter_obj.filter_by_date_and_time(date_str, start_hour, end_hour)
            filter_obj.display_results(limit=20)

        elif choice == '2':
            if filter_obj.filtered_data:
                stats = filter_obj.get_statistics()
                print("\n📊 Statistics:")
                for key, value in stats.items():
                    print(f"   • {key}: {value}")
            else:
                print("❌ No filtered data. Filter data first.")

        elif choice == '3':
            if filter_obj.filtered_data:
                output_file = input("Enter output CSV filename (default: filtered_data.csv): ").strip()
                output_file = output_file if output_file else "filtered_data.csv"
                filter_obj.export_to_csv(output_file)
            else:
                print("❌ No filtered data to export.")

        elif choice == '4':
            if filter_obj.filtered_data:
                output_file = input("Enter output JSON filename (default: filtered_data.json): ").strip()
                output_file = output_file if output_file else "filtered_data.json"
                filter_obj.export_to_json(output_file)
            else:
                print("❌ No filtered data to export.")

        elif choice == '5':
            print("👋 Goodbye!")
            break

        else:
            print("❌ Invalid option. Please select 1-5.")


# Example usage for scripting (non-interactive)
def example_usage():
    """Example of how to use the filter in scripts."""

    # Create filter instance
    filter_obj = IrrigationDataFilter("Quan trắc thực địa (1).json")

    # Filter data for a specific date (5 AM to 11:59 PM)
    filtered = filter_obj.filter_by_date_and_time("2025-02-18", start_hour=5, end_hour=23)

    print(f"\n✅ Filtered {len(filtered)} records")

    # Get statistics
    stats = filter_obj.get_statistics()
    print(f"\n📊 Statistics: {stats}")

    # Display first 10 records
    filter_obj.display_results(limit=10)

    # Export results
    filter_obj.export_to_csv("irrigation_filtered_data.csv")
    filter_obj.export_to_json("irrigation_filtered_data.json")


if __name__ == "__main__":
    # Uncomment one of the following:

    # For interactive menu:
    main()

    # For scripting example:
    # example_usage()

🌾 Irrigation Monitoring Data Filter
✅ Loaded 61112 records from Quan trắc thực địa (1).json

📋 Menu:
  1. Filter by date and time
  2. View statistics
  3. Export to CSV
  4. Export to JSON
  5. Exit


KeyboardInterrupt: Interrupted by user